In [6]:
import synapseclient
import pyarrow as pa
from pyarrow import fs
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import re
from collections import defaultdict
import time
# import duckdb
# import rpy2
# import rpy2.robjects as robjects
# %load_ext rpy2.ipython

########
# Set up Access and download dataset
########
syn = synapseclient.Synapse()
syn.login()

ARCHIVE_VERSION = '2024-05-21'
COHORT = 'adults'
##just make sure that all of them are in one cohort as initially we are considering the dataset at cohort level
## since we have all data partitioned as ARCHIVE_VERSION/COHORT/DATASET_TYPE/PARTICIPANT_ID/part-x.parquet

########
#### Set up access and Get list of valid datasets
#### archived versions of the external parquet dataset (syn52506069)
########
## Set up Token access
sts_token = syn.get_sts_storage_token(entity='syn54128737', permission='read_only', output_format='json')

s3_external = fs.S3FileSystem(
    access_key=sts_token['accessKeyId'],
    secret_key=sts_token['secretAccessKey'],
    session_token=sts_token['sessionToken'],
    region='us-east-1'
)

## Get list of datasets in the S3 bucket
base_s3_uri_external = f"{sts_token['bucket']}/{sts_token['baseKey']}/{ARCHIVE_VERSION}/{COHORT}"
selector = fs.FileSelector(base_s3_uri_external, recursive=False)
parquet_datasets_external = s3_external.get_file_info(selector)

## Get all valid datasets
valid_paths = []
i = 0
for dataset in parquet_datasets_external:
    if re.search(r'recover-velsera-integration/main/archive/', dataset.path, re.IGNORECASE):
        i += 1
        print(f"{i}: {dataset.path}")
        valid_paths.append(dataset.path)

## Get dataset type (for eg., dataset_enrolledparticipants)
valid_paths_ext_df = pd.DataFrame(valid_paths, columns=['parquet_path_external'])
valid_paths_ext_df['datasetType'] = valid_paths_ext_df['parquet_path_external'].apply(lambda x: x.split('/')[5])

##############
## Read Fitbit Intraday in chunks.
## Each chunk contains all data for a select set of participants
##############
subset_paths_df = valid_paths_ext_df[valid_paths_ext_df['datasetType'] == 'dataset_fitbitsleeplogs'].reset_index()

participant_counts = defaultdict(int)

# get dataset path
path = subset_paths_df['parquet_path_external'][0]

# dataset at cohort level (eg. fitbitintradaycombined/adults/)
dataset_cohort = ds.dataset(path, filesystem=s3_external)

# Required list of participants 
## list of required participants; just make sure that all of them are in one cohort as initially we are considering
## the dataset at cohort level
# a few adult participants, some participant can have multiple parquet files associated with them
participant_list = ['RA11003-00024', 'RA11001-00250', 'RA11004-00028']
file_list = [] # initiate list of files that represent the subset of data corresponding to participant_list


# Find the required parquet paths from dataset.files, i,e just the participants we want
for file_parquet in dataset_cohort.files:
    participant_id = file_parquet.split('/')[6]
    if participant_id in participant_list:
        # print(participant_id)
        file_list.append(file_parquet)

# Create a dataset from file_list, this would correspond to only the required participants
dataset = ds.dataset(file_list, filesystem=s3_external, format='parquet')

Welcome, yj167000!




UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.4.1) is available. Your version (4.3.0) can be upgraded by typing:
    pip install --upgrade synapseclient

Python Synapse Client version 4.4.1 release notes

https://python-docs.synapse.org/news/



1: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants
2: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants_customfields_symptoms
3: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_enrolledparticipants_customfields_treatments
4: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitactivitylogs
5: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitdailydata
6: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitdevices
7: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitecg
8: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitecg_waveformsamples
9: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitintradaycombined
10: recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitrestingheartrates
11: recover-velsera-integration/main/archive/2024-05-21/ad

In [5]:
valid_paths_ext_df

,parquet_path_external,datasetType
0,recover-velsera-integration/main/archive/2024-...,dataset_enrolledparticipants
1,recover-velsera-integration/main/archive/2024-...,dataset_enrolledparticipants_customfields_symp...
2,recover-velsera-integration/main/archive/2024-...,dataset_enrolledparticipants_customfields_trea...
3,recover-velsera-integration/main/archive/2024-...,dataset_fitbitactivitylogs
4,recover-velsera-integration/main/archive/2024-...,dataset_fitbitdailydata
5,recover-velsera-integration/main/archive/2024-...,dataset_fitbitdevices
6,recover-velsera-integration/main/archive/2024-...,dataset_fitbitecg
7,recover-velsera-integration/main/archive/2024-...,dataset_fitbitecg_waveformsamples
8,recover-velsera-integration/main/archive/2024-...,dataset_fitbitintradaycombined
9,recover-velsera-integration/main/archive/2024-...,dataset_fitbitrestingheartrates


In [7]:
dataset_cohort.files

['recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00011/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00033/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00041/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00047/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00056/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00075/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00076/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fitbitsleeplogs/RA11001-00079/part-0.parquet',
 'recover-velsera-integration/main/archive/2024-05-21/adults/dataset_fit

In [3]:
len(dataset_cohort.files)

3602

In [15]:
intra_comb = dataset.to_table().to_pandas()
intra_comb['Type'].valu

Type
stages     269
classic     16
Name: count, dtype: int64

In [16]:
intra_comb.columns

Index(['ParticipantIdentifier', 'StartDate', 'EndDate', 'Duration',
       'Efficiency', 'InfoCode', 'MinutesAfterWakeup', 'MinutesAsleep',
       'MinutesAwake', 'MinutesToFallAsleep', 'TimeInBed', 'Type',
       'SleepLevelDeep', 'SleepLevelLight', 'SleepLevelRem', 'SleepLevelWake',
       'SleepLogDetails', 'LogId', 'ParticipantID', 'IsMainSleep', 'LogType',
       'export_start_date', 'export_end_date', 'SleepLevelAwake',
       'SleepLevelAsleep', 'SleepLevelRestless', 'ModifiedDate', 'cohort'],
      dtype='object')

In [22]:
intra_comb['TimeInBed']

0      514
1      495
2      488
3      274
4      466
      ... 
280    454
281    630
282    275
283    411
284    165
Name: TimeInBed, Length: 285, dtype: object

In [23]:
valid_paths_ext_df['datasetType']

0                          dataset_enrolledparticipants
1     dataset_enrolledparticipants_customfields_symp...
2     dataset_enrolledparticipants_customfields_trea...
3                            dataset_fitbitactivitylogs
4                               dataset_fitbitdailydata
5                                 dataset_fitbitdevices
6                                     dataset_fitbitecg
7                     dataset_fitbitecg_waveformsamples
8                        dataset_fitbitintradaycombined
9                       dataset_fitbitrestingheartrates
10                              dataset_fitbitsleeplogs
11              dataset_fitbitsleeplogs_sleeplogdetails
12                             dataset_googlefitsamples
13                 dataset_healthkitv2activitysummaries
14                 dataset_healthkitv2electrocardiogram
15      dataset_healthkitv2electrocardiogram_subsamples
16                         dataset_healthkitv2heartbeat
17              dataset_healthkitv2heartbeat_sub